# Model comparison: KS tests and AIC

Compares WITCH fits of the same data, ie: MOOJ1142 with r_s fixed, r_s free, r_s plus alpha or gamma free,
and the full gNFW shape free.

**Method**
* Each run is loaded from its output directory (`config.yaml`, `results_mcmc.dill`, `samples_mcmc.npz`).
* Every pair of runs is checked to be **comparable**, `ComparabilityError` is raised if not:
  same data and likelihood config, same structures and ties, and parameters fixed in both have the same value.
  This is all the AIC needs, so non-nested runs (ie: alpha free vs gamma free) can be compared with it.
* A pair is **nested** if, in addition, the smaller run's free parameters are a strict subset of the larger run's
  and every parameter only the larger run frees is fixed (in the smaller run) inside the larger run's prior.
* **KS:** for each nested pair, a two-sample KS test on the posterior of each shared free parameter. The p-value uses the effective
  sample size of the chains since MCMC samples are correlated, and a `ChainQualityError` is raised if any
  parameter has fewer than `MIN_ESS` effective samples.
* **AIC:** read from the `aic.yaml` each fit writes at the end (`witch.fitter.aic_refit`). The fit re-estimates the
  noise from its own residuals, so its chi-squared isn't comparable between models. `aic.yaml` instead holds the
  chi-squared under **one noise model shared by every fit of the same data, the one estimated from the raw data before
  any model subtraction**, both at the starting point (the MCMC medians) and after an LM refit from there (the maximum
  likelihood, which the AIC is defined at). AIC = chisq + 2k with k the number of free parameters.

Runs that predate `aic_refit` need one made with `sbatch scripts/aic_from_run.sbatch <run outdir>` (temporary script).
Nothing here loads the data, so the notebook runs anywhere in seconds.


In [ ]:
# Runs to compare, label -> output directory. Order doesn't matter, they are sorted by number of free parameters.
REDUCTIONS = "/scratch/jorlo/MUSTANG/Reductions/MOOJ1142"
RUNS = {
    "fixed_rs": f"{REDUCTIONS}/cluster_model_sph_isobeta-gnfw_rs+ps_model_ps_gauss/xray_peak_rs/r-beta-amp-P0-beta-dx_g-dy_g-sigma-amp_g",
    "free_rs": f"{REDUCTIONS}/cluster_model_sph_isobeta-gnfw_rs+ps_model_ps_gauss/xray_peak_rs/r-beta-amp-P0-r_s-beta-dx_g-dy_g-sigma-amp_g",
    "free_alpha": f"{REDUCTIONS}/mustang2/cluster_model_sph_isobeta-gnfw_rs+ps_model_ps_gauss/xray_peak_rs_free_alpha/r-beta-amp-P0-r_s-alpha-beta-dx_g-dy_g-sigma-amp_g",
    # P0, r_s, alpha free with beta fixed to 2 x the X-ray value (4.1): same k as free_rs, nested in free_alpha
    "free_rs_alpha": f"{REDUCTIONS}/mustang2/cluster_model_sph_isobeta-gnfw_rs+ps_model_ps_gauss/xray_peak_rs_free_rs_alpha/r-beta-amp-P0-r_s-alpha-dx_g-dy_g-sigma-amp_g",
    "free_gamma": f"{REDUCTIONS}/mustang2/cluster_model_sph_isobeta-gnfw_rs+ps_model_ps_gauss/xray_peak_rs_free_gamma/r-beta-amp-P0-r_s-gamma-beta-dx_g-dy_g-sigma-amp_g",
    # 10000 step run, not converged (Rhat up to ~2): use its refit AIC only, not its KS tests
    "all_free_10k": f"{REDUCTIONS}/mustang2/cluster_model_sph_isobeta-gnfw_rs+ps_model_ps_gauss/xray_peak_rs_all_free/r-beta-amp-P0-r_s-gamma-alpha-beta-dx_g-dy_g-sigma-amp_g",
    "all_free": f"{REDUCTIONS}/mustang2/cluster_model_sph_isobeta-gnfw_rs+ps_model_ps_gauss/xray_peak_rs_all_free_80k/r-beta-amp-P0-r_s-gamma-alpha-beta-dx_g-dy_g-sigma-amp_g",
}
MIN_ESS = 50  # Minimum effective samples per parameter for the KS test
FIGDIR = "figures"


In [ ]:
import os

os.environ.setdefault("JAX_PLATFORMS", "cpu")
os.makedirs(FIGDIR, exist_ok=True)

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from witch import model_comparison as wmc

pd.set_option("display.float_format", lambda x: f"{x:.6g}")


In [ ]:
runs = []
for label, outdir in RUNS.items():
    if not os.path.isfile(os.path.join(outdir, "samples_mcmc.npz")):
        print(f"Skipping {label}, no finished MCMC in {outdir}")
        continue
    runs.append(wmc.load_run(outdir, label))
# Raises ComparabilityError if any pair can't be compared
runs = wmc.sort_comparable(runs)
display(pd.DataFrame([{"run": r.label, "k": r.k, "chains": r.chains.shape, "outdir": r.outdir} for r in runs]))
pairs = wmc.nested_pairs(runs)
for small, big in pairs:
    print(f"{small.label} nested in {big.label}, which also frees {wmc.check_nested(small, big)}")


## Chain convergence

ESS is the effective number of independent samples, `tau` the autocorrelation time in steps, and `Rhat` the split
Gelman-Rubin statistic. Rhat above ~1.1 means the chains disagree: that run's medians and KS results are not
reliable and it needs a longer chain. The refit AIC only uses the medians as a starting point, so it is still usable.


In [ ]:
diag = pd.DataFrame([row for run in runs for row in wmc.chain_diagnostics(run)])
summary = diag.groupby("run", sort=False).agg(min_ESS=("ESS", "min"), max_tau=("tau", "max"), max_Rhat=("Rhat", "max"))
display(summary)
for label, row in summary.iterrows():
    if row["max_Rhat"] > 1.1 or row["min_ESS"] < MIN_ESS:
        print(f"WARNING {label} is not converged, worst parameters:")
        display(diag[diag["run"] == label].sort_values("Rhat", ascending=False).head(5))


## KS tests on the shared free parameters of each nested pair

Pairs where a chain has fewer than `MIN_ESS` effective samples in a shared parameter are skipped
(`ks_compare` raises `ChainQualityError`, reported here) rather than stopping the notebook.

`D` is the KS statistic (max distance between the CDFs), `p` the ESS corrected p-value. A small `p` means the
marginal posterior of that parameter moved when the extra parameters were freed; this says nothing about which model
fits better (that is what the AIC below is for).


In [ ]:
ks_tables = {}
for small, big in pairs:
    try:
        ks_tables[(small.label, big.label)] = df = pd.DataFrame(wmc.ks_compare(small, big, MIN_ESS))
    except wmc.ChainQualityError as e:
        print(f"SKIPPED {small.label} vs {big.label}: {e}")
        continue
    print(f"{small.label} vs {big.label}")
    display(df)


In [ ]:
shared = [n for n in runs[0].free_names if all(n in r.free_names for r in runs)]  # Free in every run
ncol = 3
fig, axes = plt.subplots(int(np.ceil(len(shared) / ncol)), ncol, figsize=(4 * ncol, 3 * np.ceil(len(shared) / ncol)))
for ax, name in zip(axes.flat, shared):
    for run in runs:
        ax.hist(run.chains[..., run.free_names.index(name)].ravel(), bins=60, density=True, histtype="step", label=run.label)
    ax.set_xlabel(name.split(".", 1)[1])
for ax in list(axes.flat)[len(shared):]:
    ax.set_visible(False)
axes.flat[0].legend()
fig.tight_layout()
fig.savefig(os.path.join(FIGDIR, "nested_posteriors.png"), dpi=150)


## AIC under the shared initial noise model

From each run's `aic.yaml`:
* **start:** chi-squared at the starting point, the MCMC medians.
* **refit:** after the LM refit from there, the maximum likelihood. Prefer this one: the medians can sit off the ridge
  of a degenerate posterior (worst for the models with the most free parameters) and come from chains run with each
  model's own re-estimated noise; the refit removes both.

`shift` is how far each parameter moved in the refit in units of its posterior standard deviation; shifts of a few
sigma mean the median was a poor point estimate of the best fit.

In [ ]:
missing = [run.label for run in runs if run.aic is None]
for label in missing:
    print(f"No aic.yaml for {label}, make one with: sbatch scripts/aic_from_run.sbatch {RUNS[label]}")
aic_runs = [run for run in runs if run.aic is not None]
if len(aic_runs) < 2:
    print("Need aic.yaml for at least two runs to compare")
    aic = {}
else:
    display(pd.DataFrame([
        {"run": run.label, "start": run.aic["start"], **{f"lm_{k}": v for k, v in run.aic["lm"].items()}}
        for run in aic_runs
    ]))
    shift = {}
    for run in aic_runs:
        flat = run.chains.reshape(-1, run.k)
        # aic.yaml lists the free parameters in parameter order, the same order as the chain columns
        refit = np.array([par["refit"] for par in run.aic["parameters"]])
        shift[run.label] = dict(zip(run.free_names, (refit - np.median(flat, axis=0)) / np.std(flat, axis=0)))
    display(pd.DataFrame(shift).rename_axis("shift / posterior std"))

    aic = {}
    for which in ("start", "refit"):
        chisqs = wmc.aic_chisqs(aic_runs, which)
        for warning in wmc.chisq_order_warnings(aic_runs, chisqs):
            print(f"WARNING ({which}), larger model fits worse:", warning)
        aic[which] = pd.DataFrame(wmc.aic_table(aic_runs, chisqs))
        print(which)
        display(aic[which])

In [ ]:
if aic:
    fig, ax = plt.subplots(figsize=(6, 3))
    width = 0.8 / len(aic)
    x = np.arange(len(aic_runs))
    for i, (which, df) in enumerate(aic.items()):
        ax.bar(x + (i - (len(aic) - 1) / 2) * width, df["dAIC"], width, label=which)
    ax.set_xticks(x, [run.label for run in aic_runs])
    ax.axhline(10, color="k", ls="--", lw=1, label=r"$\Delta$AIC = 10")
    ax.set_ylabel(r"$\Delta$AIC")
    ax.legend()
    fig.tight_layout()
    fig.savefig(os.path.join(FIGDIR, "nested_daic.png"), dpi=150)

## Reading the results
* ΔAIC is relative to the best (lowest AIC) run. ΔAIC > 10 means essentially no support for that model,
  2–10 is positive to strong evidence against it, < 2 means the models are indistinguishable.
* Prefer the **refit** AIC, it is the formal definition. If it differs a lot from the start AIC for a model,
  that model's posterior is degenerate enough that its medians are a poor point estimate.
* At the maximum likelihood a larger model can never fit worse than a model nested in it. If the refit
  still prints a "larger model fits worse" warning, its LM refit didn't reach the maximum
  (check `lm_lambda` above, and raise `fitting.maxiter` for that run).

To make an HTML report: `jupyter nbconvert --to html --execute notebooks/nested_model_comparison.ipynb`
